# Train the climbing-hold detector — one action

**To train:**
1. Edit the **Config** cell once (your Drive path, epochs, etc.).
2. **Runtime → Run all.** That's it.

**If Colab disconnects mid-training:** just **Runtime → Run all** again.
The driver auto-detects the interrupted run and **resumes** it — no cell to
choose, nothing to redo. (It knows a run is unfinished because finished runs
write a `run_meta.json`; a killed one doesn't.)

**Before the first run** — on your Drive `DRIVE_DIR` put:
- `dataset.zip` (zip root holds `train/ val/ test/`) and `dataset.yaml`
  (both from `scripts/prepare_dataset.py`).
- Runtime → Change runtime type → **GPU**.

Adding new data later? Re-upload `dataset.zip`/`dataset.yaml`, then Run all —
it starts a fresh run on the bigger dataset automatically.

In [ ]:
# ====== CONFIG — edit these, then Runtime ▸ Run all ======
REPO_URL  = 'https://github.com/antonylgs/yolov11-climbing-holds-detector.git'
DRIVE_DIR = '/content/drive/MyDrive/climbing-holds-detector'  # holds dataset.zip + dataset.yaml + runs/
EPOCHS    = 150
IMGSZ     = 1280
BATCH     = 16        # safe at imgsz 1280 on A100; bump to 24/32 if epoch 1 has VRAM headroom
PATIENCE  = 30        # early-stop if val mAP doesn't improve for this many epochs
RUN_NAME  = 'holds'


In [ ]:
# ====== DRIVER — setup, train (auto fresh/resume), results. Don't edit. ======
import os, glob, pathlib, zipfile, shutil, subprocess

# 1) Mount Drive (safe to re-run)
from google.colab import drive
drive.mount('/content/drive')

DATA_YAML, DATASET_ZIP = f'{DRIVE_DIR}/dataset.yaml', f'{DRIVE_DIR}/dataset.zip'
DATASET, RUN_DIR       = f'{DRIVE_DIR}/data/dataset', f'{DRIVE_DIR}/runs'
LOCAL_DATASET, LOCAL_YAML, REPO = '/content/dataset', '/content/dataset.yaml', '/content/climbing-holds-detector'
assert os.path.exists(DATA_YAML), f'missing {DATA_YAML} on Drive'
assert os.path.exists(DATASET_ZIP) or os.path.isdir(DATASET), f'upload dataset.zip to {DRIVE_DIR}'

# 2) Code + deps (clone or pull, then install the package)
subprocess.run(['git','clone',REPO_URL,REPO] if not os.path.isdir(REPO)
               else ['git','-C',REPO,'pull','--ff-only'], check=True)
os.chdir(REPO)
subprocess.run(['pip','install','-q','-e','.'], check=True)

# 3) Stage dataset on fast local disk (skip if present) + point yaml at it
if not os.path.isdir(LOCAL_DATASET):
    if os.path.exists(DATASET_ZIP):
        print('unzipping dataset…'); zipfile.ZipFile(DATASET_ZIP).extractall(LOCAL_DATASET)
    else:
        print('copying dataset…'); shutil.copytree(DATASET, LOCAL_DATASET)
assert os.path.isdir(f'{LOCAL_DATASET}/train/images'), 'unexpected dataset layout'
pathlib.Path(LOCAL_YAML).write_text('\n'.join(
    f'path: {LOCAL_DATASET}' if ln.startswith('path:') else ln
    for ln in pathlib.Path(DATA_YAML).read_text().splitlines()) + '\n')

# 4) Auto-decide: resume an interrupted run, else start fresh.
#    Finished run -> has run_meta.json. Killed run -> last.pt but no run_meta.json.
def unfinished_run():
    for last in sorted(glob.glob(f'{RUN_DIR}/{RUN_NAME}*/weights/last.pt'),
                       key=os.path.getmtime, reverse=True):
        run = os.path.dirname(os.path.dirname(last))
        if not os.path.exists(f'{run}/run_meta.json'):
            return last
    return None

resume = unfinished_run()
if resume:
    print(f'\u21bb Resuming interrupted run:\n   {resume}\n')
    cmd = f'python scripts/train.py --resume "{resume}" --project "{RUN_DIR}" --no-export'
else:
    print('\u25b6 Starting a fresh run\n')
    cmd = (f'python scripts/train.py --data "{LOCAL_YAML}" --epochs {EPOCHS} --imgsz {IMGSZ} '
           f'--batch {BATCH} --patience {PATIENCE} --project "{RUN_DIR}" --name {RUN_NAME} --no-export')
subprocess.run(cmd, shell=True)

# 5) Results (runs persist on Drive)
hits = sorted(glob.glob(f'{RUN_DIR}/**/weights/best.pt', recursive=True), key=os.path.getmtime)
if hits:
    run = os.path.dirname(os.path.dirname(hits[-1]))
    print('\n\u2705 best weights :', hits[-1])
    print('   result curves:', f'{run}/results.png')
    print('   Download best.pt into the repo as models/best.pt to run the CLI locally.')
